In [1]:
from openvino import Core
core = Core()
print(core.available_devices)
# Should print: ['CPU', 'GPU', 'NPU']


['CPU', 'GPU', 'NPU']


In [5]:
from optimum.intel import OVModelForCausalLM
from transformers import AutoTokenizer
import openvino as ov

model_path = "../models/phi3.5-mini-ov-int4"

# 1. Load without compiling
model = OVModelForCausalLM.from_pretrained(
    model_path,
    device="GPU",
    compile=False,
    ov_config={"CACHE_DIR": "./ov_cache"}
)

# 2. MANUALLY SET INPUT TYPES TO I32
#    This modifies the OpenVINO graph in-memory to expect 32-bit integers
#    instead of 64-bit, satisfying the NPU requirement.
for item in model.model.inputs:
    if item.get_element_type() == ov.Type.i64:
        item.get_node().set_element_type(ov.Type.i32)
        # Also need to ensure the tensor names/layouts align if this was more complex,
        # but for changing precision usually this is enough or we use PrePostProcessor.

# Alternative: Use PrePostProcessor if the direct set fails (which it might on frozen nodes)
# But let's try the reshape+compile first, as sometimes reshape triggers type recalculation if we are lucky.

# 3. Resize and Compile
model.reshape(1, 128)
model.compile()

tokenizer = AutoTokenizer.from_pretrained(model_path)
input_text = "Explain transformer attention:"
inputs = tokenizer(input_text, return_tensors="pt")

# IMPORTANT: Cast PyTorch inputs to int32 before generating!
inputs = {k: v.to(dtype=int) for k, v in inputs.items()} # .to(int) in PyTorch is int32 usually

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Static shapes are not supported for causal language model.


Explain transformer attention:

Transformer attention, also known as self-attention or intra-attention, is a mechanism that allows each position in a sequence to attend to all positions within the same sequence to compute a representation of the sequence. It is a


In [ ]:
# With sym input shapes and LLMPipeline
from openvino_genai import LLMPipeline

# Define configuration to enable model caching
# This will create a folder 'model_cache' to store the compiled NPU model
config = { "CACHE_DIR": "./model_cache" }

# Load and run on Intel NPU with caching enabled
# Note: The FIRST run will still take ~1 minute to compile and save the cache.
#       All SUBSEQUENT runs will be fast (loading from disk).
pipe = LLMPipeline(
    "../models/phi-4-mini-instruct-int4-sym", 
    device="CPU", 
    **config
)

# Generate text
response = pipe.generate("Explain quantum computing:", max_new_tokens=)
print(response)

Quantum computing is a rapidly advancing field of technology that leverages the perplexing and counterintuitive principles of quantum mechanics to process information in ways that traditional computers cannot. At the heart of quantum computing are quantum bits, or qubits, which are the basic units of quantum information. Unlike classical bits, which can be either 0 or 1, qubits can exist in a state of 0, 1, or any quantum superposition of these states. This means they can perform many calculations


In [11]:
import time
from openvino_genai import LLMPipeline

model_path = "../models/phi-4-mini-instruct-int4-sym"
config = { "CACHE_DIR": "./model_cache" }

def run_benchmark(device, prompt, max_tokens, label):
    print(f"\n--- {label} [{device}] ---")
    
    # Measure Load Time
    t0 = time.time()
    pipe = LLMPipeline(model_path, device=device, **config)
    t1 = time.time()
    print(f"Load Time: {t1-t0:.2f}s")
    
    # Warmup (optional, to settle clock speeds)
    pipe.generate("Warmup", max_new_tokens=1)
    
    # Measure Generation Time
    start = time.time()
    response = pipe.generate(prompt, max_new_tokens=max_tokens)
    end = time.time()
    
    total_time = end - start
    # Approximate perf (this mixes prefill and decode, but good for overall feel)
    print(f"Inference Time: {total_time:.2f}s")
    del pipe

# Scenario 1: Heavy Prefill (Input Scale)
# NPU should theoretically outperform CPU here
long_prompt = "Explain the history of the Roman Empire in great detail, focusing on: " * 20 
run_benchmark("CPU", long_prompt, 10, "Hevy Input (Prefill)")
run_benchmark("NPU", long_prompt, 10, "Heavy Input (Prefill)")

# Scenario 2: Heavy Generation (Output Scale)
# Both likely constrained by Memory Bandwidth
short_prompt = "Count to 100: "
run_benchmark("CPU", short_prompt, 150, "Long Generation (Decode)")
run_benchmark("NPU", short_prompt, 150, "Long Generation (Decode)")


--- Hevy Input (Prefill) [CPU] ---
Load Time: 0.69s
Inference Time: 2.70s

--- Heavy Input (Prefill) [NPU] ---
Load Time: 3.12s
Inference Time: 1.13s

--- Long Generation (Decode) [CPU] ---
Load Time: 0.63s
Inference Time: 6.19s

--- Long Generation (Decode) [NPU] ---
Load Time: 3.08s
Inference Time: 7.65s
